# CosySim Gemma 270M Router Fine-Tuning

Fine-tunes `google/gemma-3-270m-it` with Unsloth QLoRA for:
- **Tag extraction** — parse [MOOD:x] [IMAGE:x] from LLM output
- **Tool routing** — classify intent → tool call
- **Priority classify** — request → priority tier
- **Decision classify** — NPC state → action
- **Response validation** — check output format

**Requirements**: Google Colab Pro (T4 GPU), ~5 compute units per run

**Upload** your `training/datasets/*.jsonl` files to Google Drive first.

In [ ]:
# Cell 1: Install Unsloth + dependencies
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes xformers

In [ ]:
# Cell 2: Configuration
import os

# === EDIT THESE ===
DATASET_NAME = "tag_extraction"  # Change per training run
DRIVE_PATH = "/content/drive/MyDrive/cosysim_training"  # Where your JSONL files are
OUTPUT_DIR = f"/content/cosysim-gemma-{DATASET_NAME}"
HF_REPO = None  # Set to "username/cosysim-gemma-router" to push to HuggingFace

# Training hyperparameters
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_SEQ_LENGTH = 2048

print(f"Training: {DATASET_NAME}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Get training data — choose ONE method

# METHOD A: Clone the CosySim repo directly (recommended with VS Code Colab)
import subprocess, os
if not os.path.isdir('/content/CosySim'):
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none',
                    '--sparse', 'https://github.com/nihilistau/CosySim.git',
                    '/content/CosySim'], check=True)
    subprocess.run(['git', 'sparse-checkout', 'set', 'training/datasets'],
                   cwd='/content/CosySim', check=True)
train_path = f'/content/CosySim/training/datasets/{DATASET_NAME}_train.jsonl'
val_path = f'/content/CosySim/training/datasets/{DATASET_NAME}_val.jsonl'

# METHOD B: Google Drive (uncomment these, comment out METHOD A)
# from google.colab import drive
# drive.mount('/content/drive')
# train_path = f'{DRIVE_PATH}/{DATASET_NAME}_train.jsonl'
# val_path = f'{DRIVE_PATH}/{DATASET_NAME}_val.jsonl'

assert os.path.exists(train_path), f'Train file not found: {train_path}'
print(f'Train: {train_path}')
print(f'Val:   {val_path}')

In [ ]:
# Cell 4: Load base model with 4-bit quantization
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3-270m-it",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Cell 5: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Cell 6: Load and format training data
import json
from datasets import Dataset

# Qwen3-style tool-call template for Gemma
CHAT_TEMPLATE = """<start_of_turn>user
{instruction}

Input: {input}<end_of_turn>
<start_of_turn>model
{output}<end_of_turn>"""

def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

def format_example(ex):
    return {"text": CHAT_TEMPLATE.format(**ex)}

train_raw = load_jsonl(train_path)
val_raw = load_jsonl(val_path)

train_ds = Dataset.from_list([format_example(ex) for ex in train_raw])
val_ds = Dataset.from_list([format_example(ex) for ex in val_raw])

print(f"Train: {len(train_ds)} examples")
print(f"Val:   {len(val_ds)} examples")
print(f"\nSample:\n{train_ds[0]['text'][:500]}")

In [ ]:
# Cell 7: Configure trainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,  # Pack short sequences together
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not __import__('torch').cuda.is_bf16_supported(),
        bf16=__import__('torch').cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
)

print("Trainer configured. Ready to train.")

In [ ]:
# Cell 8: Train!
import time

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final loss: {stats.training_loss:.4f}")

In [ ]:
# Cell 9: Quick evaluation
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Test with a sample from validation set
test_examples = [
    # Tag extraction
    "She smiled [MOOD:happy] and took a selfie [IMAGE:a warm selfie] before sitting [ACTION:sit_down]",
    # Tool routing
    "I want to know how lola feels about me",
    # Priority
    "bedroom_scene: character responding to player speech",
]

for test_input in test_examples:
    prompt = f"<start_of_turn>user\nProcess this input:\n\nInput: {test_input}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Input:  {test_input[:80]}")
    print(f"Output: {result.split('model')[-1][:200]}")
    print("---")

In [ ]:
# Cell 10: Evaluate with evaluate_model() from finetune_local.py
# Upload finetune_local.py and generate_datasets.py to Colab, or clone the repo.
import sys, os

# Make CosySim importable if the repo was cloned into /content
COSYSIM_ROOT = "/content/CosySim"  # adjust if you cloned elsewhere
if os.path.isdir(COSYSIM_ROOT) and COSYSIM_ROOT not in sys.path:
    sys.path.insert(0, COSYSIM_ROOT)

from training.finetune_local import evaluate_model

adapter_path = f"{OUTPUT_DIR}/lora"
model.save_pretrained(adapter_path)  # save current adapter before eval
tokenizer.save_pretrained(adapter_path)

# Datasets produced by generate_datasets.py
DATASETS = [
    "tag_extraction",
    "tool_routing",
    "priority_classify",
    "decision_classify",
    "response_validate",
]

# Evaluate on the dataset we just trained on
result = evaluate_model(
    adapter_path=adapter_path,
    dataset_name=DATASET_NAME,
    max_samples=50,
)
print(f"Dataset:  {result['dataset']}")
print(f"Accuracy: {result['correct']}/{result['total']} ({result['accuracy']:.1%})")
print(f"\nAll 5 datasets: {', '.join(DATASETS)}")

In [ ]:
# Cell 11: Export GGUF (for LMStudio)
# Q4_K_M — small, fast (~150MB)
model.save_pretrained_gguf(
    f"{OUTPUT_DIR}/gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Q4_K_M saved to {OUTPUT_DIR}/gguf/")

# Q8_0 — higher quality (~300MB)
model.save_pretrained_gguf(
    f"{OUTPUT_DIR}/gguf-q8",
    tokenizer,
    quantization_method="q8_0",
)
print(f"Q8_0 saved to {OUTPUT_DIR}/gguf-q8/")

In [ ]:
# Cell 12: Save LoRA adapter (for further iteration)
model.save_pretrained(f"{OUTPUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
print(f"LoRA adapter saved to {OUTPUT_DIR}/lora/")

In [ ]:
# Cell 13: Copy to Google Drive for download
import shutil

drive_out = f"{DRIVE_PATH}/models/{DATASET_NAME}"
os.makedirs(drive_out, exist_ok=True)

# Copy GGUF files
for src_dir in [f"{OUTPUT_DIR}/gguf", f"{OUTPUT_DIR}/gguf-q8"]:
    if os.path.exists(src_dir):
        dest = f"{drive_out}/{os.path.basename(src_dir)}"
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(src_dir, dest)

# Copy LoRA
lora_dest = f"{drive_out}/lora"
if os.path.exists(lora_dest):
    shutil.rmtree(lora_dest)
shutil.copytree(f"{OUTPUT_DIR}/lora", lora_dest)

print(f"All outputs copied to {drive_out}/")
print("Download from Google Drive to your local machine.")

In [ ]:
# Cell 14: (Optional) Push to HuggingFace Hub
if HF_REPO:
    model.push_to_hub_gguf(
        HF_REPO,
        tokenizer,
        quantization_method=["q4_k_m", "q8_0"],
    )
    print(f"Pushed to https://huggingface.co/{HF_REPO}")
else:
    print("Set HF_REPO to push to HuggingFace Hub.")

## Next Steps

1. Download the GGUF from Google Drive
2. Copy to LMStudio's models directory
3. Load as T3 router model in CosySim config
4. Test via the admin panel

To train the next dataset, change `DATASET_NAME` in Cell 2 and re-run from Cell 6.

**Training order**: tag_extraction → tool_routing → priority_classify → decision_classify → response_validate